In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

In [ ]:
df= pd.read_csv('/content/Titanic-Dataset.xls')

In [ ]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [ ]:
df1 = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [ ]:
X_train, X_test, y_train, y_Test = train_test_split(df1.drop('Survived',axis=1),df1['Survived'],test_size=0.2,random_state=42)

In [ ]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


# *Imputation Transform*

In [ ]:
trn_miss_val = ColumnTransformer([
    ('impute_age', SimpleImputer(),[2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'),[6])],
     remainder='passthrough')

In [ ]:
trn_ohe = ColumnTransformer([
    ('ohe',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
],remainder='passthrough')

In [ ]:
trn_scaling = ColumnTransformer([
    ('s_scaler',StandardScaler(),slice(0,10))
])

In [ ]:
trn_deci_tree = DecisionTreeClassifier()
trn_log_reg = LogisticRegression()
trn_random_forest = RandomForestClassifier()
# trn_lr = LinearRegression()

## Create Pipeline

In [ ]:
pipe_tree = Pipeline([
    ('missing_value',trn_miss_val),
    ('ohe',trn_ohe),
    ('scaling',trn_scaling),
    ('decision_tree',trn_deci_tree)
])

In [ ]:
pipe_log_reg = Pipeline([
    ('missing_value',trn_miss_val),
    ('ohe',trn_ohe),
    ('scaling',trn_scaling),
    ('Logistic_reg',trn_log_reg)
])

In [ ]:
pipe_random_forest = Pipeline([
    ('missing_value',trn_miss_val),
    ('ohe',trn_ohe),
    ('scaling',trn_scaling),
    ('random_forest',trn_random_forest)
])

In [ ]:
pipe_log_reg.fit(X_train,y_train)

Pipeline(steps=[('missing_value',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ohe',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('scaling',
                 ColumnTransformer(transformers=[('s_scaler', StandardScaler(),
                                                  slice(0, 10, None))])),
                ('Logistic_reg', LogisticRegression())])

In [ ]:
pipe_random_forest.fit(X_train,y_train)

Pipeline(steps=[('missing_value',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ohe',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('scaling',
                 ColumnTransformer(transformers=[('s_scaler', StandardScaler(),
                                                  slice(0, 10, None))])),
                ('random_forest', RandomForestClassifier())])

In [ ]:
pipe_tree.fit(X_train,y_train)

Pipeline(steps=[('missing_value',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ohe',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('scaling',
                 ColumnTransformer(transformers=[('s_scaler', StandardScaler(),
                                                  slice(0, 10, None))])),
                ('decision_tree', DecisionTreeClassifier())])

In [ ]:
y_pred_log_reg = pipe_log_reg.predict(X_test)

In [ ]:
y_pred_tree = pipe_tree.predict(X_test)

In [ ]:
y_pred_forest = pipe_random_forest.predict(X_test)

In [ ]:
tree_accuracy = accuracy_score(y_Test,y_pred_tree)
tree_accuracy

0.6256983240223464

In [ ]:
forest_accuracy = accuracy_score(y_Test,y_pred_forest)
forest_accuracy

0.6256983240223464

In [ ]:
log_reg_accuracy = accuracy_score(y_Test,y_pred_log_reg)
log_reg_accuracy

0.6256983240223464

In [ ]:
from sklearn import set_config
set_config(display='diagram')

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe_log_reg,X_train,y_train,cv=5,scoring='accuracy').mean()

np.float64(0.6391214419383433)

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe_tree,X_train,y_train,cv=5,scoring='accuracy').mean()

np.float64(0.6391214419383433)

In [ ]:
params={
    'decision_tree__max_depth':[1,2,3,4,5,None]
}

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe_tree,params,cv=5,scoring='accuracy')

In [ ]:
grid.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('missing_value',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('ohe',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('scaling',
                                        ColumnTransformer(transformers=[('s_scaler',
                                                                         StandardScaler(),
                                                                         slice(0, 10, None))])),
                                       ('decision_tree',
                                        DecisionTreeClassifier())]),
             param_grid={'decision_tree__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [ ]:
grid.best_score_

np.float64(0.6391214419383433)

In [ ]:
grid.best_params_

{'decision_tree__max_depth': 2}

In [ ]:
import pickle

In [ ]:
pickle.dump(pipe_tree,open('model_tree.pkl','wb'))
pickle.dump(pipe_random_forest,open('model_forest.pkl','wb'))
pickle.dump(pipe_log_reg,open('model_log_reg.pkl','wb'))

In [ ]:
X_test

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
709,3,male,NaN,1,1,15.2458,C
439,2,male,31.0,0,0,10.5000,S
840,3,male,20.0,0,0,7.9250,S
720,2,female,6.0,0,1,33.0000,S
39,3,female,14.0,1,0,11.2417,C
...,...,...,...,...,...,...,...
433,3,male,17.0,0,0,7.1250,S
773,3,male,NaN,0,0,7.2250,C
25,3,female,38.0,1,5,31.3875,S
84,2,female,17.0,0,0,10.5000,S
